# 🦀 CRust — C-to-Rust Migration RL Training
**Meta OpenEnv Hackathon | Theme #2: Super Long-Horizon Planning & Instruction Following**

Trains **Qwen 2.5 3B** (or Llama 3 8B) via **GRPO + Unsloth** to migrate legacy C code to memory-safe, modular Rust.

| Component | Where it runs |
|---|---|
| **OpenEnv Server** (for judges) | HF Space A10G — `https://adithyakommuri-meta-hackathon-final.hf.space` |
| **GRPO Training** | This Colab (T4 GPU, free) |
| **Reward function** | Local CRust env inside Colab (fast, no HTTP latency) |

**⏱ Expected time**: ~45 min for 100 steps on T4 · ~20 min on A10G

## Cell 1 — Install everything (run once, ~8 min)

In [ ]:
import os, subprocess, sys

# ── 1a. Install Unsloth + TRL (handles torch/peft/transformers compatibility) ──
print('Installing Python packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'unsloth', 'trl>=0.8.0', 'datasets', 'requests', 'matplotlib', '-q'],
               check=True)

# ── 1b. Install Rust (needed for cargo check / cargo test in verifier) ──
print('Installing Rust toolchain...')
subprocess.run('curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --no-modify-path',
               shell=True, check=True)

# Add cargo to PATH for this session
cargo_bin = os.path.expanduser('~/.cargo/bin')
os.environ['PATH'] = cargo_bin + ':' + os.environ.get('PATH', '')

# Verify
result = subprocess.run(['cargo', '--version'], capture_output=True, text=True)
print('Rust ready:', result.stdout.strip())
print('\n✅ All dependencies installed!')

## Cell 2 — Clone CRust project from GitHub

In [ ]:
import os, sys

# Clone the project from GitHub
GITHUB_REPO = 'https://github.com/22adi66/meta_pytorch_scalar_hackathon.git'
PROJECT_DIR = '/content/crust_project'

if not os.path.exists(PROJECT_DIR):
    os.system(f'git clone {GITHUB_REPO} {PROJECT_DIR}')
else:
    os.system(f'cd {PROJECT_DIR} && git pull')

# Add to Python path
sys.path.insert(0, PROJECT_DIR)

# Point workspace at the Rust dummy_workspace inside the project
WORKSPACE = f'{PROJECT_DIR}/crust_env/dummy_workspace'
os.environ['CRUST_WORKSPACE'] = WORKSPACE

# Also add cargo to PATH (persists across cells)
cargo_bin = os.path.expanduser('~/.cargo/bin')
os.environ['PATH'] = cargo_bin + ':' + os.environ.get('PATH', '')

print('Project path:', PROJECT_DIR)
print('Workspace:', WORKSPACE)
print('Files:', os.listdir(f'{PROJECT_DIR}/crust_env'))

## Cell 3 — Verify the CRust environment works locally

In [ ]:
import json
from crust_env.env import MigrationEnv

env = MigrationEnv(workspace_dir=os.environ['CRUST_WORKSPACE'])

# Reset Phase 1 — should return math_ops.c or similar leaf node
obs = env.reset(phase=1)
print('=== Environment Reset OK ===')
print('Target file  :', obs['current_target'])
print('Constraints  :', obs['constraints'])
print('Phase        :', obs['phase'])
print('Files remain :', obs['files_remaining'])
print()
print('C source code preview:')
print(obs['c_source_code'][:500])

In [ ]:
# Verify reward function works end-to-end
# Test 1: bad code → should get low reward
obs = env.reset(phase=1)
bad = env.step({'file_path': 'src/math_ops.rs', 'code_content': 'fn main() {}'})
print(f'Bad code reward : {bad["reward"]}   stage={bad["info"]["verification"]["stage"]}')

# Test 2: correct Rust → should get high reward
obs = env.reset(phase=1)
good_rust = '''
pub fn add(a: i32, b: i32) -> i32 { a + b }
pub fn subtract(a: i32, b: i32) -> i32 { a - b }
pub fn multiply(a: i32, b: i32) -> i32 { a * b }
pub fn divide(a: i32, b: i32) -> Option<i32> {
    if b == 0 { None } else { Some(a / b) }
}
pub fn clamp(value: i32, min_val: i32, max_val: i32) -> i32 {
    value.max(min_val).min(max_val)
}
'''
good = env.step({'file_path': 'src/math_ops.rs', 'code_content': good_rust})
print(f'Good code reward: {good["reward"]}   stage={good["info"]["verification"]["stage"]}')
print(f'Breakdown: {json.dumps(good["info"]["reward_breakdown"], indent=2)}')
print()
print('✅ Environment + reward function verified!')

## Cell 4 — Load model with Unsloth (QLoRA 4-bit)

In [ ]:
import torch
from unsloth import FastLanguageModel

# Qwen 2.5 3B — fits on T4 (16GB) with 4-bit  ← FREE Colab
# Switch to Llama-3-8B-Instruct on A10G for better results
MODEL_NAME     = 'unsloth/Qwen2.5-3B-Instruct'
MAX_SEQ_LENGTH = 2048

print(f'Loading {MODEL_NAME} in 4-bit QLoRA...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,       # auto: bf16 on Ampere, fp16 otherwise
    load_in_4bit  = True,
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

vram = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM used: {vram:.1f} GB / {total:.1f} GB')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print('✅ Model ready!')

## Cell 5 — Build dataset and reward function

In [ ]:
import re
from datasets import Dataset
from crust_env.env import MigrationEnv

SYSTEM_PROMPT = """You are an expert Rust systems programmer specializing in memory-safe C-to-Rust migration.

Translate the given C source file to idiomatic, safe Rust. Rules:
1. NEVER use `unsafe` — use Rust ownership/borrowing instead.
2. CBO (external crate imports) must be below 3.
3. Every public function must be semantically equivalent to its C counterpart.
4. Use idiomatic Rust: Option<T>, iterators, match expressions.
5. Output ONLY the complete .rs file. No markdown, no explanations."""

def build_prompt(obs):
    constraints = '\n'.join(f'  - {c}' for c in obs.get('constraints', []))
    dep_ctx = obs.get('dependency_context', {})
    dep_str = '\n'.join(f'// {f}:\n{s}' for f,s in dep_ctx.items()) or '// No prior translations yet.'
    errors  = obs.get('recent_errors', [])
    err_str = '\n'.join(f'  [{e.get("level")}] {e.get("message")}' for e in errors) or '  None'

    user_msg = (
        f"## Migration Task\nTranslate this C file to memory-safe Rust.\n\n"
        f"## Constraints\n{constraints}\n\n"
        f"## C Source: {obs.get('current_target')}\n```c\n{obs.get('c_source_code','')}\n```\n\n"
        f"## Already-Translated Context\n{dep_str}\n\n"
        f"## Recent Compiler Errors (fix these)\n{err_str}\n\n"
        f"Provide the complete Rust source file:"
    )
    return tokenizer.apply_chat_template(
        [{'role':'system','content':SYSTEM_PROMPT},
         {'role':'user',  'content':user_msg}],
        tokenize=False, add_generation_prompt=True
    )

# Build dataset across multiple constraint combinations (instruction following diversity)
def make_dataset(phase=1):
    env = MigrationEnv(workspace_dir=os.environ['CRUST_WORKSPACE'])
    prompts = []
    for c_set in [
        ['Do not use the unsafe keyword', 'Maintain a CBO score below 3'],
        ['Do not use the unsafe keyword'],
        ['Maintain a CBO score below 3'],
        ['Do not use the unsafe keyword', 'Maintain a CBO score below 3'],
    ]:
        obs = env.reset(constraints=c_set, phase=phase)
        if obs.get('current_target'):
            prompts.append(build_prompt(obs))
    return Dataset.from_dict({'prompt': prompts})

dataset = make_dataset(phase=1)
print(f'Dataset: {len(dataset)} prompts')
print(f'Sample prompt length: {len(dataset[0]["prompt"])} chars')
print('\n=== Sample prompt preview (first 600 chars) ===')
print(dataset[0]['prompt'][:600])

In [ ]:
# Reward function — fully programmatic (no LLM judge)
def reward_func(prompts, completions, **kwargs):
    rewards = []
    env = MigrationEnv(workspace_dir=os.environ['CRUST_WORKSPACE'])
    for completion in completions:
        try:
            obs = env.reset(phase=1)
            target = obs.get('current_target', 'math_ops.c')
            rs_path = 'src/' + re.sub(r'\.c$', '.rs', os.path.basename(target))
            result = env.step({'file_path': rs_path, 'code_content': completion})
            rewards.append(result['reward'])
        except Exception as e:
            print(f'[reward_func error] {e}')
            rewards.append(0.01)
    return rewards

# Quick sanity check
test_r = reward_func(prompts=['test'], completions=['pub fn add(a: i32, b: i32) -> i32 { a + b }'])
print(f'Sanity check reward: {test_r[0]}')
print('✅ Reward function ready!')

## Cell 6 — 📸 Measure BASELINE (before training) — save this!

In [ ]:
import json

FastLanguageModel.for_inference(model)

def generate(prompt, max_new_tokens=400):
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    input_text = tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)
    return decoded[len(input_text):].strip()

env = MigrationEnv(workspace_dir=os.environ['CRUST_WORKSPACE'])
sample_obs = env.reset(phase=1)
sample_prompt = build_prompt(sample_obs)

print('===============================')
print('  BASELINE (zero-shot model)  ')
print('===============================')

baseline_rewards = []
baseline_stages  = []

for i in range(4):   # 4 independent runs
    completion = generate(sample_prompt)
    obs2 = env.reset(phase=1)
    r = env.step({'file_path': 'src/math_ops.rs', 'code_content': completion})
    baseline_rewards.append(r['reward'])
    baseline_stages.append(r['info']['verification']['stage'])
    print(f'  Run {i+1}: reward={r["reward"]:.4f}  stage={r["info"]["verification"]["stage"]}')

baseline_avg = sum(baseline_rewards) / len(baseline_rewards)
print(f'\n📊 Baseline average reward: {baseline_avg:.4f}')
print('\n⚠️  SCREENSHOT THIS — you need it for the before/after comparison!')

FastLanguageModel.for_training(model)

## Cell 7 — 🚀 GRPO Training (main training loop)

In [ ]:
from trl import GRPOTrainer, GRPOConfig

# ── Config ──────────────────────────────────────────────────────────────────
# T4  (16GB): batch=1, grad_accum=4, steps=100  → ~45 min
# A10G(24GB): batch=2, grad_accum=4, steps=200  → ~40 min

IS_A10G = torch.cuda.get_device_properties(0).total_memory > 20e9
BATCH   = 2 if IS_A10G else 1
STEPS   = 200 if IS_A10G else 100

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Using batch_size={BATCH}, max_steps={STEPS}')

training_args = GRPOConfig(
    output_dir                 = '/content/crust_outputs',
    learning_rate              = 2e-5,
    per_device_train_batch_size= BATCH,
    gradient_accumulation_steps= 4,
    max_steps                  = STEPS,
    logging_steps              = 5,
    save_steps                 = 50,
    warmup_ratio               = 0.05,
    lr_scheduler_type          = 'cosine',
    num_generations            = 4,      # GRPO group size — 4 rollouts per prompt
    temperature                = 0.8,
    max_new_tokens             = 512,
    report_to                  = 'none', # change to 'wandb' for tracking
    seed                       = 42,
)

trainer = GRPOTrainer(
    model           = model,
    processing_class= tokenizer,
    reward_funcs    = reward_func,   # 100% programmatic — no LLM judge
    args            = training_args,
    train_dataset   = dataset,
)

print(f'\n🚀 Starting GRPO training ({STEPS} steps)...')
print('Watch the REWARD column — it should trend upward!')
print('─' * 60)

trainer.train()

print('─' * 60)
print('✅ Training complete!')

## Cell 8 — 📊 Measure AFTER training + generate comparison chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

FastLanguageModel.for_inference(model)

env = MigrationEnv(workspace_dir=os.environ['CRUST_WORKSPACE'])

print('===============================')
print('  TRAINED MODEL (post-GRPO)   ')
print('===============================')

trained_rewards = []
trained_stages  = []

for i in range(4):
    completion = generate(sample_prompt)
    obs2 = env.reset(phase=1)
    r = env.step({'file_path': 'src/math_ops.rs', 'code_content': completion})
    trained_rewards.append(r['reward'])
    trained_stages.append(r['info']['verification']['stage'])
    print(f'  Run {i+1}: reward={r["reward"]:.4f}  stage={r["info"]["verification"]["stage"]}')

trained_avg = sum(trained_rewards) / len(trained_rewards)

print(f'\n📊 Results:')
print(f'  Baseline avg : {baseline_avg:.4f}')
print(f'  Trained avg  : {trained_avg:.4f}')
delta = trained_avg - baseline_avg
pct   = (delta / max(baseline_avg, 0.01)) * 100
print(f'  Improvement  : +{delta:.4f}  ({pct:+.1f}%)')

# ── Before/After Bar Chart (for hackathon submission) ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CRust: GRPO Training Results — Meta OpenEnv Hackathon', fontsize=14, fontweight='bold')

# Bar chart
ax = axes[0]
bars = ax.bar(
    ['Baseline\n(zero-shot)', 'CRust GRPO\n(trained)'],
    [baseline_avg, trained_avg],
    color=['#e74c3c', '#2ecc71'], width=0.5, edgecolor='white', linewidth=2
)
ax.set_ylabel('Average Reward (0–1)', fontsize=12)
ax.set_title('Reward Before vs After GRPO', fontsize=12)
ax.set_ylim(0, 1.0)
ax.axhline(y=0.40, color='orange', linestyle='--', alpha=0.6, label='Compiles (0.40)')
ax.axhline(y=0.99, color='green',  linestyle='--', alpha=0.4, label='All tests pass (0.99)')
ax.legend(fontsize=9)
for bar, val in zip(bars, [baseline_avg, trained_avg]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}',
            ha='center', va='bottom', fontweight='bold', fontsize=13)

# Stage breakdown
ax2 = axes[1]
stage_labels = ['compilation_fail', 'compilation', 'testing', 'complete']
colors_map   = {'compilation_fail':'#e74c3c','compilation':'#e67e22','testing':'#f1c40f','complete':'#2ecc71'}
all_stages   = baseline_stages + trained_stages
groups       = ['Baseline']*len(baseline_stages) + ['Trained']*len(trained_stages)

b_counts = {s: baseline_stages.count(s) for s in set(baseline_stages + trained_stages)}
t_counts = {s: trained_stages.count(s)  for s in set(baseline_stages + trained_stages)}
all_s    = sorted(set(baseline_stages + trained_stages))
x        = range(len(all_s))
ax2.bar([i - 0.2 for i in x], [b_counts.get(s,0) for s in all_s], width=0.35,
        color='#e74c3c', label='Baseline', alpha=0.85)
ax2.bar([i + 0.2 for i in x], [t_counts.get(s,0) for s in all_s], width=0.35,
        color='#2ecc71', label='Trained', alpha=0.85)
ax2.set_xticks(list(x))
ax2.set_xticklabels(all_s, rotation=15)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Verification Stage Distribution', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.savefig('/content/crust_reward_improvement.png', dpi=180, bbox_inches='tight')
plt.show()
print('\n💾 Chart saved to /content/crust_reward_improvement.png')
print('📌 Download this image — it is your main hackathon proof of improvement!')

## Cell 9 — 🔍 Side-by-side demo (for judges)

In [ ]:
print('=' * 70)
print('  JUDGE DEMO: Trained Model Generating a Rust Migration')
print('=' * 70)

env = MigrationEnv(workspace_dir=os.environ['CRUST_WORKSPACE'])
obs = env.reset(phase=1, constraints=['Do not use the unsafe keyword', 'Maintain a CBO score below 3'])

print(f'\nTask: Translate {obs["current_target"]} → Rust')
print(f'Constraints: {obs["constraints"]}')
print(f'\nC Source:\n{obs["c_source_code"]}\n')
print('─' * 70)
print('Generated Rust (trained model):')
print('─' * 70)

prompt = build_prompt(obs)
rust_code = generate(prompt, max_new_tokens=600)
print(rust_code)

print('\n─' * 70)
result = env.step({'file_path': 'src/math_ops.rs', 'code_content': rust_code})
print(f'\n🎯 Reward: {result["reward"]}   Stage: {result["info"]["verification"]["stage"]}')
print(f'📐 Metrics: {result["info"]["metrics"]}')
print(f'💰 Breakdown: {json.dumps(result["info"]["reward_breakdown"], indent=2)}')

## Cell 10 — 💾 Save model + push to HF Hub

In [ ]:
from huggingface_hub import login

HF_TOKEN    = 'YOUR_HF_TOKEN_HERE'  # paste your token from https://huggingface.co/settings/tokens
HF_USERNAME = 'Adithyakommuri'
REPO_NAME   = f'{HF_USERNAME}/crust-grpo-qwen25-3b'

login(token=HF_TOKEN, add_to_git_credential=False)

# IMPORTANT: Use Unsloth's proper merge path — do NOT naively upcast 4-bit to 16-bit!
print('Saving LoRA adapters...')
model.save_pretrained('/content/crust_lora')
tokenizer.save_pretrained('/content/crust_lora')

# Push LoRA adapters to HF Hub (small upload ~100MB)
print(f'Pushing to {REPO_NAME}...')
model.push_to_hub(REPO_NAME, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_NAME, token=HF_TOKEN)

print(f'\n✅ Model saved to: https://huggingface.co/{REPO_NAME}')
print('Include this URL in your hackathon submission!')

## Cell 11 — 🌐 Verify the live HF Space OpenEnv server

In [ ]:
import requests, json

BASE = 'https://adithyakommuri-meta-hackathon-final.hf.space'

print(f'Testing live OpenEnv server at {BASE}...')
print()

# Health
h = requests.get(f'{BASE}/health', timeout=15)
print('Health:', h.json())

# Info
info = requests.get(f'{BASE}/info', timeout=15)
print('\nEnvironment info:')
print(json.dumps(info.json(), indent=2)[:600])

# Reset
obs = requests.post(f'{BASE}/reset', json={'phase': 1}, timeout=30).json()
print(f'\nReset OK — target: {obs.get("current_target")}')
print(f'Constraints: {obs.get("constraints")}')

# Step (submit our trained output to the live server)
final_result = requests.post(f'{BASE}/step', json={
    'file_path': 'src/math_ops.rs',
    'code_content': rust_code   # from Cell 9
}, timeout=60).json()

print(f'\n🎯 Live server reward: {final_result["reward"]}')
print(f'Stage: {final_result["info"]["verification"]["stage"]}')
print(f'\n✅ Live HF Space OpenEnv server is working!')
print(f'🔗 Share with judges: {BASE}/docs')